# Чтение векторных данных


> 🚧 Раздел в разработке.


В предыдущем разделе мы познакомились с моделями пространственных данных, рассмотрели принципы их описания и создали набор пространственных данных в формате **GeoDataFrame** на основе координат точки. Однако на практике чаще всего мы работаем с готовыми наборами данных.


> В этом разделе рассмотрим основные форматы хранения пространственных данных, способы их импорта


## 0. Импорт библиотек


In [ ]:
import pandas as pd
import geopandas as gpd

- [**pandas**](https://pandas.pydata.org/) (`pandas`) — это библиотека Python для работы с табличными данными. Она предоставляет удобные структуры данных (DataFrame и Series) и инструменты для очистки, анализа, преобразования и агрегации данных.

- [**GeoPandas**](https://geopandas.org/) (`geopandas`) — это библиотека Python для работы с векторными пространственными данными. Она расширяет возможности pandas, добавляя поддержку геометрии.


## 1. Форматы векторных данных


Для хранения пространственных векторных данных есть спциальные файловые форматы. Рассмотрим наиболее распространённые их них:

1. **Shapefile (.shp)**

Формат, разработанный компанией Esri. Один из самых распространённых форматов в ГИС. Данные хранятся в виде набора связанных файлов (.shp, .shx, .dbf). Несмотря на широкую поддержку, имеет технические ограничения.

2. **GeoJSON (.geojson)**

Текстовый формат на основе JSON, широко используемый в веб-картографии. Удобен для обмена данными и хорошо читается, но не очень эффективен для больших объёмов данных.

3. **GeoPackage (.gpkg)**

Открытый стандарт, разработанный Open Geospatial Consortium. По сути представляет базу данных и позволяет хранить несколько разные наборы данных в одном файле. Основан на SQLite.


Для загрузки пространственных данных используется функция `read_file()` библиотеки **GeoPandas**, которая позволяет импортировать данные различных векторных форматов.

Рассмотрим пример импорта данных различных форматов.

В примерах будут использованы файлы из нашего репозитория, однако при желании вы можете работать со своими данными аналогичных форматов.


### 1.1. Shapefile


Для этого формата важно помнить, что это **не один файл**, а набор связанных файлов. Для корректного чтения данных минимально необходимы:

- `.shp` — геометрия
- `.dbf` — атрибуты
- `.shx` — индекс
- `.prj` — система координат

Хотя при чтении данных мы указываем путь только к `.shp`, все остальные файлы набора должны находиться в той же директории.


In [ ]:
gdf_shp = gpd.read_file("data/spb_dtp_shp/spb_dtp.shp")

Выведем первые десять строк атрибутивной таблицы с помощью метода `head()`:


In [ ]:
gdf_shp.head()

Посмотрим на данные на карте c помощью метода `explore()`:


In [ ]:
gdf_shp.explore(tiles='cartodbpositron')

### 1.2. GeoJSON


В отличие от Shapefile, формат **GeoJSON** представляет собой **один файл**, в котором одновременно хранятся геометрия объектов, их атрибуты и сведения о системе координат.

Поэтому для чтения данных достаточно, чтобы файл `.geojson` находился в нужной директории.


In [ ]:
gdf_geojson = gpd.read_file("data/spb_metro.geojson") 

Выведем первые десять строк атрибутивной таблицы с помощью метода `head()`:


In [ ]:
gdf_geojson.head()

Посмотрим на данные на карте c помощью метода `explore()`:


In [ ]:
gdf_geojson.explore(tiles='cartodbpositron')

### 1.3. GeoPackage


Для этого формата важно помнить, что внутри одного файла `.gpkg` могут храниться несколько наборов данных (слоёв). Если это так, при чтении необходимо указать, какой именно слой вы хотите загрузить, иначе по умолчанию будет прочитан первый доступный слой.


Чтобы узнать, какие слои содержатся в файле `.gpkg`, можно воспользоваться функцией `list_layers()` из GeoPandas.


In [ ]:
gpd.list_layers("data/spb_admin.gpkg")

Эта команда вернула список названий слоёв, один из которых теперь можно указать в параметре `layer`


In [ ]:
gdf_gpkg = gpd.read_file("data/spb_admin.gpkg", layer="okrug")

Выведем первые десять строк атрибутивной таблицы с помощью метода `head()`:


In [ ]:
gdf_gpkg.head()

Посмотрим на данные на карте c помощью метода `explore()`:


In [ ]:
gdf_gpkg.explore(tiles='cartodbpositron')

## 2.Табличные данные


Иногда пространственная информация хранится в виде координат и не содержит явной геометрии. Например, часто мы сталкиваемся с табличными данными, в которых для каждого объекта указаны долгота и широта.

В таком виде данные сами по себе не являются пространственными. Однако на основе координат можно создать набор пространственных данных, преобразовав, сформировав геометрию на основе координт и создав объект типа **GeoDataFrame**.

Один из наиболее распространённых форматов хранения табличных данных — **CSV (Comma-Separated Values)** — это простой текстовый формат хранения табличных данных, в котором значения в строках разделяются специальным символом (обычно запятой).

Рассмотрим как на основе табличных данных с координатами создать набор пространственных данных


### 2.1. Чтение CSV


Загрузим CSV-файл с помощью функции `read_csv()` библиотеки **pandas**:


In [ ]:
df_csv = pd.read_csv('./data/spb_theaters.csv')

Выведем первые строки таблицы с помощью метода `head()`, чтобы ознакомиться со структурой данных и определить, в каких полях содержатся координаты


In [ ]:
df_csv.head()

### 2.2. Создание GeoDataFrame


Преобразуем координаты из столбцов `latitude` и `longitude` в геометрию точек с помощью функции `points_from_xy()` библиотеки GeoPandas:


In [ ]:
df_csv = df_csv.dropna(subset=["longitude", "latitude"]) #удалим строки с пустыми координатами

geometry = gpd.points_from_xy(df_csv["longitude"], df_csv["latitude"])

Создадим `GeoDataFrame`, добавив сформированную геометрию к исходной таблице:


In [ ]:
gdf_csv = gpd.GeoDataFrame(df_csv, geometry=geometry, crs="EPSG:4326")

Здесь мы:

- передаём исходную таблицу `df_csv`,
- добавляем созданную геометрию,
- указываем систему координат (`EPSG:4326` — широта и долгота).


Посмотрим на данные на карте c помощью метода `explore()`:


In [ ]:
gdf_csv.head()

In [ ]:
gdf_csv.explore(tiles='cartodbpositron')

### 2.3. Сохранение результата


После создания `GeoDataFrame` мы можем сохранить полученный набор пространственных данных в любом поддерживаемом формате. Для этого используется метод `to_file()` библиотеки GeoPandas.

Например:


In [ ]:
gdf_csv.to_file("output_data.gpkg", driver="GPKG")

Таким образом, созданные нами данные можно сохранить в формате **GeoPackage**, **Shapefile**, **GeoJSON** и других поддерживаемых форматах.


## 3. Итог


В этом разделе мы познакомились с основными форматами хранения векторных пространственных данных и научились импортировать их в Python с помощью библиотеки **GeoPandas**.

Мы рассмотрели особенности форматов **Shapefile**, **GeoJSON** и **GeoPackage**, а также разобрали различия в их структуре и способах чтения.

Кроме того, мы изучили, как работать с табличными данными в формате **CSV**, создавать геометрию на основе координат и формировать объект типа **GeoDataFrame**.
